###  Importing libraries 

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer

### Loading the datasets

In [2]:
def load_splits():
    """
    Load train, validation, and test datasets.
    """
    train_df = pd.read_csv("train.csv")
    val_df = pd.read_csv("validation.csv")
    test_df = pd.read_csv("test.csv")

    return train_df, val_df, test_df

### Separate labels and features

In [3]:
def get_features_and_labels(df):
    """
    Extract features (X) and labels (y) from dataframe.
    """
    X = df["clean_message"]
    y = df["label"]
    return X, y

### Preparing the features

In [4]:
def fit_vectorizer(X_train, max_features=5000):
    """
    Fit TF-IDF vectorizer on training data only.
    """
    vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=(1, 2)
    )
    X_train_vec = vectorizer.fit_transform(X_train)

    return vectorizer, X_train_vec

In [5]:
def transform_data(vectorizer, X):
    """
    Transform data using a fitted vectorizer.
    """
    return vectorizer.transform(X)

In [6]:
def drop_missing_text(df):
    """
    Remove rows where clean_message is missing or empty.
    """
    before = len(df)
    df = df.dropna(subset=["clean_message"])
    df = df[df["clean_message"].str.strip() != ""]
    after = len(df)

    print(f"Dropped {before - after} rows with empty clean_message")

    return df

In [7]:
def prepare_training_data():
    train_df, val_df, test_df = load_splits()

    train_df = drop_missing_text(train_df)
    val_df = drop_missing_text(val_df)
    test_df = drop_missing_text(test_df)

    X_train, y_train = get_features_and_labels(train_df)
    X_val, y_val = get_features_and_labels(val_df)
    X_test, y_test = get_features_and_labels(test_df)

    vectorizer, X_train_vec = fit_vectorizer(X_train)
    X_val_vec = transform_data(vectorizer, X_val)
    X_test_vec = transform_data(vectorizer, X_test)

    return (
        X_train_vec, y_train,
        X_val_vec, y_val,
        X_test_vec, y_test,
        vectorizer
    )


In [8]:
X_train_vec, y_train, X_val_vec, y_val, X_test_vec, y_test, vectorizer = prepare_training_data()

X_train_vec.shape, X_val_vec.shape, X_test_vec.shape

Dropped 3 rows with empty clean_message
Dropped 1 rows with empty clean_message
Dropped 2 rows with empty clean_message


((3615, 5000), (774, 5000), (774, 5000))

### Model functions

In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [10]:
def fit_model(model, X_train, y_train):
    """
    Fit a classification model on training data.
    """
    model.fit(X_train, y_train)
    return model

In [11]:
def score_model(model, X):
    """
    Generate predictions using a trained model.
    """
    return model.predict(X)

In [12]:
def evaluate_predictions(y_true, y_pred):
    """
    Evaluate classification predictions.
    """
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "confusion_matrix": confusion_matrix(y_true, y_pred)
    }

    return metrics

In [13]:
def print_evaluation(y_true, y_pred, split_name=""):
    """
    Print detailed evaluation results.
    """
    print(f"\n--- {split_name} Evaluation ---")
    print(classification_report(y_true, y_pred, target_names=["ham", "spam"]))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

In [14]:
def validate_model(model, X_train, y_train, X_val, y_val):
    """
    Fit model and evaluate on train and validation sets.
    """
    model = fit_model(model, X_train, y_train)

    # Train evaluation
    y_train_pred = score_model(model, X_train)
    print_evaluation(y_train, y_train_pred, "Train")

    # Validation evaluation
    y_val_pred = score_model(model, X_val)
    print_evaluation(y_val, y_val_pred, "Validation")

    return model


---

### Model 1- Multionomial Naive Bayes

In [15]:
from sklearn.naive_bayes import MultinomialNB

In [16]:
def get_mnb_model(alpha=1.0):
    """
    Create a Multinomial Naive Bayes model.
    """
    return MultinomialNB(alpha=alpha)

In [17]:
mnb_model = get_mnb_model()
mnb_model = validate_model(
    mnb_model,
    X_train_vec, y_train,
    X_val_vec, y_val
)


--- Train Evaluation ---
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99      3158
        spam       1.00      0.83      0.91       457

    accuracy                           0.98      3615
   macro avg       0.99      0.92      0.95      3615
weighted avg       0.98      0.98      0.98      3615

Confusion Matrix:
[[3158    0]
 [  76  381]]

--- Validation Evaluation ---
              precision    recall  f1-score   support

         ham       0.95      1.00      0.98       676
        spam       1.00      0.66      0.80        98

    accuracy                           0.96       774
   macro avg       0.98      0.83      0.89       774
weighted avg       0.96      0.96      0.95       774

Confusion Matrix:
[[676   0]
 [ 33  65]]


high precision, low recall behavior.

### hyperparameter tuning

In [18]:
def tune_mnb_alpha(X_train, y_train, X_val, y_val, alphas):
    """
    Tune alpha for MultinomialNB using validation data.
    """
    results = []

    for alpha in alphas:
        model = MultinomialNB(alpha=alpha)
        model.fit(X_train, y_train)

        y_val_pred = model.predict(X_val)
        metrics = evaluate_predictions(y_val, y_val_pred)

        results.append({
            "alpha": alpha,
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"]
        })

    return pd.DataFrame(results)

In [19]:
alphas = [0.01, 0.05, 0.1, 0.5, 1.0]
mnb_tuning_results = tune_mnb_alpha(
    X_train_vec, y_train,
    X_val_vec, y_val,
    alphas
)

mnb_tuning_results

,alpha,precision,recall,f1
0,0.01,0.930233,0.816327,0.869565
1,0.05,0.952381,0.816327,0.879121
2,0.10,0.963415,0.806122,0.877778
3,0.50,1.000000,0.755102,0.860465
4,1.00,1.000000,0.663265,0.797546


In [20]:
best_alpha = 0.05 

best_mnb_model = MultinomialNB(alpha=best_alpha)
best_mnb_model = validate_model(
    best_mnb_model,
    X_train_vec, y_train,
    X_val_vec, y_val
)



--- Train Evaluation ---
              precision    recall  f1-score   support

         ham       0.99      1.00      1.00      3158
        spam       1.00      0.95      0.97       457

    accuracy                           0.99      3615
   macro avg       0.99      0.97      0.98      3615
weighted avg       0.99      0.99      0.99      3615

Confusion Matrix:
[[3156    2]
 [  25  432]]

--- Validation Evaluation ---
              precision    recall  f1-score   support

         ham       0.97      0.99      0.98       676
        spam       0.95      0.82      0.88        98

    accuracy                           0.97       774
   macro avg       0.96      0.91      0.93       774
weighted avg       0.97      0.97      0.97       774

Confusion Matrix:
[[672   4]
 [ 18  80]]


---

### Model 2 - Logistic Regression

In [21]:
from sklearn.linear_model import LogisticRegression

In [22]:
def get_logistic_model(C=1.0):
    """
    Create a Logistic Regression classifier.
    """
    return LogisticRegression(
        C=C,
        solver="liblinear",
        max_iter=1000,
        class_weight=None
    )

In [23]:
lr_model = get_logistic_model()
lr_model = validate_model(
    lr_model,
    X_train_vec, y_train,
    X_val_vec, y_val
)


--- Train Evaluation ---
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98      3158
        spam       0.99      0.73      0.84       457

    accuracy                           0.97      3615
   macro avg       0.98      0.87      0.91      3615
weighted avg       0.97      0.97      0.96      3615

Confusion Matrix:
[[3154    4]
 [ 122  335]]

--- Validation Evaluation ---
              precision    recall  f1-score   support

         ham       0.95      0.99      0.97       676
        spam       0.94      0.62      0.75        98

    accuracy                           0.95       774
   macro avg       0.94      0.81      0.86       774
weighted avg       0.95      0.95      0.94       774

Confusion Matrix:
[[672   4]
 [ 37  61]]


### hyperparameter tuning

In [24]:
def tune_logistic_C(X_train, y_train, X_val, y_val, C_values):
    """
    Tune C for Logistic Regression using validation data.
    """
    results = []

    for C in C_values:
        model = LogisticRegression(
            C=C,
            solver="liblinear",
            max_iter=1000
        )
        model.fit(X_train, y_train)

        y_val_pred = model.predict(X_val)
        metrics = evaluate_predictions(y_val, y_val_pred)

        results.append({
            "C": C,
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"]
        })

    return pd.DataFrame(results)

In [25]:
C_values = [0.01, 0.1, 1.0, 10.0, 100.0]

lr_tuning_results = tune_logistic_C(
    X_train_vec, y_train,
    X_val_vec, y_val,
    C_values
)

lr_tuning_results

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,C,precision,recall,f1
0,0.01,0.000000,0.000000,0.000000
1,0.10,0.000000,0.000000,0.000000
2,1.00,0.938462,0.622449,0.748466
3,10.00,0.926829,0.775510,0.844444
4,100.00,0.927711,0.785714,0.850829


In [27]:
best_lr_model = get_logistic_model(C=10.0)

best_lr_model = validate_model(
    best_lr_model,
    X_train_vec, y_train,
    X_val_vec, y_val
)


--- Train Evaluation ---
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00      3158
        spam       1.00      0.98      0.99       457

    accuracy                           1.00      3615
   macro avg       1.00      0.99      0.99      3615
weighted avg       1.00      1.00      1.00      3615

Confusion Matrix:
[[3158    0]
 [  10  447]]

--- Validation Evaluation ---
              precision    recall  f1-score   support

         ham       0.97      0.99      0.98       676
        spam       0.93      0.78      0.84        98

    accuracy                           0.96       774
   macro avg       0.95      0.88      0.91       774
weighted avg       0.96      0.96      0.96       774

Confusion Matrix:
[[670   6]
 [ 22  76]]


---

## Model 3 - Linear SVM

In [28]:
from sklearn.svm import LinearSVC

In [29]:
def get_linear_svm(C=1.0):
    """
    Create a Linear SVM classifier.
    """
    return LinearSVC(
        C=C,
        max_iter=5000
    )

In [30]:
svm_model = get_linear_svm(C=1.0)

svm_model = validate_model(
    svm_model,
    X_train_vec, y_train,
    X_val_vec, y_val
)


--- Train Evaluation ---
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00      3158
        spam       1.00      0.99      0.99       457

    accuracy                           1.00      3615
   macro avg       1.00      0.99      1.00      3615
weighted avg       1.00      1.00      1.00      3615

Confusion Matrix:
[[3158    0]
 [   6  451]]

--- Validation Evaluation ---
              precision    recall  f1-score   support

         ham       0.97      0.99      0.98       676
        spam       0.94      0.80      0.86        98

    accuracy                           0.97       774
   macro avg       0.96      0.89      0.92       774
weighted avg       0.97      0.97      0.97       774

Confusion Matrix:
[[671   5]
 [ 20  78]]


### Hyperparameter Tuning

In [31]:
def tune_svm_C(X_train, y_train, X_val, y_val, C_values):
    """
    Tune C for Linear SVM using validation data.
    """
    results = []

    for C in C_values:
        model = LinearSVC(C=C, max_iter=5000)
        model.fit(X_train, y_train)

        y_val_pred = model.predict(X_val)
        metrics = evaluate_predictions(y_val, y_val_pred)

        results.append({
            "C": C,
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"]
        })

    return pd.DataFrame(results)


In [32]:
C_values = [0.01, 0.1, 1.0, 10.0]

svm_tuning_results = tune_svm_C(
    X_train_vec, y_train,
    X_val_vec, y_val,
    C_values
)

svm_tuning_results


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,C,precision,recall,f1
0,0.01,0.000000,0.000000,0.000000
1,0.10,0.942029,0.663265,0.778443
2,1.00,0.939759,0.795918,0.861878
3,10.00,0.920455,0.826531,0.870968


In [ ]:
best_svm_C = 1  

best_svm_model = get_linear_svm(C=best_svm_C)

best_svm_model = validate_model(
    best_svm_model,
    X_train_vec, y_train,
    X_val_vec, y_val
)


--- Train Evaluation ---
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00      3158
        spam       1.00      0.99      0.99       457

    accuracy                           1.00      3615
   macro avg       1.00      0.99      1.00      3615
weighted avg       1.00      1.00      1.00      3615

Confusion Matrix:
[[3158    0]
 [   6  451]]

--- Validation Evaluation ---
              precision    recall  f1-score   support

         ham       0.97      0.99      0.98       676
        spam       0.94      0.80      0.86        98

    accuracy                           0.97       774
   macro avg       0.96      0.89      0.92       774
weighted avg       0.97      0.97      0.97       774

Confusion Matrix:
[[671   5]
 [ 20  78]]


---

##  Testing the models

In [36]:
def evaluate_on_test(model, X_test, y_test, model_name):
    """
    Evaluate a trained model on the test set.
    """
    y_test_pred = model.predict(X_test)

    print(f"\n===== {model_name} | Test Evaluation =====")
    print(classification_report(y_test, y_test_pred, target_names=["ham", "spam"]))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_test_pred))

    return evaluate_predictions(y_test, y_test_pred)


In [37]:
mnb_test_metrics = evaluate_on_test(
    best_mnb_model,
    X_test_vec, y_test,
    "Multinomial Naive Bayes"
)

lr_test_metrics = evaluate_on_test(
    best_lr_model,
    X_test_vec, y_test,
    "Logistic Regression"
)

svm_test_metrics = evaluate_on_test(
    best_svm_model,
    X_test_vec, y_test,
    "Linear SVM"
)



===== Multinomial Naive Bayes | Test Evaluation =====
              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       676
        spam       0.93      0.87      0.90        98

    accuracy                           0.98       774
   macro avg       0.96      0.93      0.94       774
weighted avg       0.98      0.98      0.98       774

Confusion Matrix:
[[670   6]
 [ 13  85]]

===== Logistic Regression | Test Evaluation =====
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       676
        spam       1.00      0.84      0.91        98

    accuracy                           0.98       774
   macro avg       0.99      0.92      0.95       774
weighted avg       0.98      0.98      0.98       774

Confusion Matrix:
[[676   0]
 [ 16  82]]

===== Linear SVM | Test Evaluation =====
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       676
       

In [38]:
pd.DataFrame([
    {"model": "MNB", **mnb_test_metrics},
    {"model": "Logistic Regression", **lr_test_metrics},
    {"model": "Linear SVM", **svm_test_metrics}
])

,model,accuracy,precision,recall,f1,confusion_matrix
0,MNB,0.975452,0.934066,0.867347,0.899471,"[[670, 6], [13, 85]]"
1,Logistic Regression,0.979328,1.000000,0.836735,0.911111,"[[676, 0], [16, 82]]"
2,Linear SVM,0.981912,1.000000,0.857143,0.923077,"[[676, 0], [14, 84]]"


---

## Final Conclusion:- 

In this project, three benchmark models — Multinomial Naive Bayes, Logistic Regression, and Linear SVM — were trained and tuned using a validation set, and evaluated once on an unseen test set.

Since SMS spam classification is a precision-sensitive problem, avoiding false positives (legitimate messages incorrectly classified as spam) is critical, while still maintaining reasonable recall to detect spam.

On the test set, Linear SVM achieved the best overall performance with perfect precision, high recall, and the highest F1 score among all models. It also demonstrated strong generalization compared to Logistic Regression and slightly outperformed Multinomial Naive Bayes.

Therefore, **Linear SVM** was selected as the final model for SMS spam classification.